# Recommendation System - Combined Embeddings

This notebook combines semantic (BGE-M3) and structural (Node2Vec) embeddings to build a recommendation system for WikiCS articles using cosine similarity.

In [ ]:
import networkx as nx
import pickle
import pandas as pd
import numpy as np
import os
import json
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from IPython.display import display
import networkx as nx
import os
import pickle

## 1. Load Embeddings and Data

In [9]:
BGE_M3_PATH = "cap-embeddings/BAAI_bge-m3/master_embeddings.parquet"
NODE2VEC_PATH = "cap-embeddings/node2vec/master_embeddings.parquet"
DATA_PATH = "data/data-embeddings.json"

print("Loading embeddings...")
df_bge = pd.read_parquet(BGE_M3_PATH)
df_n2v = pd.read_parquet(NODE2VEC_PATH)

with open(DATA_PATH, "r") as f:
    data = json.load(f)

nodes = data["nodes"]
# Create ID -> Title mapping
id_to_title = {node["id"]: node["title"] for node in nodes}
title_to_id = {node["title"]: node["id"] for node in nodes}

print(f"BGE-M3 nodes: {len(df_bge)}")
print(f"Node2Vec nodes: {len(df_n2v)}")

# Build the graph for link-based evaluation
print("Building graph...")
node_ids_set = {node["id"]: node for node in nodes}
G = nx.DiGraph()
for n in nodes:
    G.add_node(n["id"], title=n["title"])
for n in nodes:
    src = n["id"]
    for tgt in n.get("outlinks", []):
        if tgt in node_ids_set:
            G.add_edge(src, tgt)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
# Build the graph for link-based evaluation
print("Building graph...")
node_ids_set = {node["id"]: node for node in nodes}
G = nx.DiGraph()
for n in nodes:
    G.add_node(n["id"], title=n["title"])
for n in nodes:
    src = n["id"]
    for tgt in n.get("outlinks", []):
        if tgt in node_ids_set:
            G.add_edge(src, tgt)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


Loading embeddings...
BGE-M3 nodes: 11701
Node2Vec nodes: 11701


## 2. Concatenate Embeddings

In [10]:
# Ensure they are sorted by id for easy concatenation or merge
df_bge = df_bge.sort_values("id").reset_index(drop=True)
df_n2v = df_n2v.sort_values("id").reset_index(drop=True)

# Check if IDs match exactly before concatenating directly
if (df_bge["id"] == df_n2v["id"]).all():
    combined_embeddings = []
    for e1, e2 in zip(df_bge["embedding"], df_n2v["embedding"]):
        combined_embeddings.append(np.concatenate([e1, e2]))
    
    df_combined = pd.DataFrame({
        "id": df_bge["id"],
        "embedding": combined_embeddings
    })
    print("Direct concatenation successful.")
else:
    print("IDs do not match exactly, performing inner merge...")
    df_combined = pd.merge(df_bge, df_n2v, on="id", suffixes=("_bge", "_n2v"))
    df_combined["embedding"] = df_combined.apply(lambda r: np.concatenate([r["embedding_bge"], r["embedding_n2v"]]), axis=1)
    df_combined = df_combined[["id", "embedding"]]

print(f"Combined embedding shape: {len(df_combined)} nodes, dim: {len(df_combined['embedding'].iloc[0])}")

Direct concatenation successful.
Combined embedding shape: 11701 nodes, dim: 1152


In [11]:
# Save combined embeddings
OUTPUT_DIR = "cap-embeddings/combined"
os.makedirs(OUTPUT_DIR, exist_ok=True)
df_combined.to_parquet(os.path.join(OUTPUT_DIR, "master_embeddings.parquet"), index=False)
print(f"Combined embeddings saved to {OUTPUT_DIR}")

Combined embeddings saved to cap-embeddings/combined


## 3. Cache Setup and Similarity Matrix Precomputation

Precompute the full cosine-similarity matrix and cache it so that recommendation
queries are instant (no re-computation on every call).  Also build a lookup for
whether a recommended item is already linked to the query in the graph.

In [ ]:
CACHE_DIR = "./cache/recommendation-0"
os.makedirs(CACHE_DIR, exist_ok=True)

SIM_MATRIX_FILE = os.path.join(CACHE_DIR, "similarity_matrix.pkl")

# Build id <-> index lookup from the combined df
embedding_matrix = np.stack(df_combined["embedding"].values)
node_ids_arr = df_combined["id"].values
id_to_idx = { nid: i for i, nid in enumerate(node_ids_arr) }
idx_to_id = { i: nid for i, nid in enumerate(node_ids_arr) }
n_nodes = len(node_ids_arr)

def compute_similarity_matrix():
    print("  [comp]  Computing full similarity matrix...")
    sim_matrix = cosine_similarity(embedding_matrix)
    return sim_matrix

if os.path.exists(SIM_MATRIX_FILE):
    print("  [cache] Loading similarity matrix from cache...")
    with open(SIM_MATRIX_FILE, "rb") as f:
        sim_matrix = pickle.load(f)
    print(f"  [cache] Loaded similarity matrix: shape {sim_matrix.shape}")
else:
    sim_matrix = compute_similarity_matrix()
    with open(SIM_MATRIX_FILE, "wb") as f:
        pickle.dump(sim_matrix, f)
    print(f"  [save]  Similarity matrix saved to cache.")

# Zero out self-similarity (diagonal) to avoid recommending the query itself
np.fill_diagonal(sim_matrix, 0.0)

# Create a fast set of existing edges for O(1) link lookup
existing_edges = set(G.edges())
print(f"Edge lookup set built: {len(existing_edges)} directed edges")

def is_linked(src_id, tgt_id):
    """Return True if there is a directed edge from src_id to tgt_id."""
    return (src_id, tgt_id) in existing_edges

print(f"Similarity matrix ready: {sim_matrix.shape}")
print(f"Graph edges: {G.number_of_edges()}")

# Save node mappings alongside the matrix for later use
with open(os.path.join(CACHE_DIR, "node_mappings.pkl"), "wb") as f:
    pickle.dump({
        "node_ids_arr": node_ids_arr,
        "id_to_idx": id_to_idx,
        "idx_to_id": idx_to_id,
        "id_to_title": id_to_title,
        "title_to_id": title_to_id,
    }, f)
print("Node mappings saved to cache.")

# Free memory
del df_bge, df_n2v, combined_embeddings


In [ ]:
def get_recommendations_by_title(title, top_k=10):
    """
    Get top-k recommendations for a given article title using the cached
    cosine-similarity matrix.  Returns a list of dicts with:
      - title      : recommended article title
      - similarity : cosine similarity score
      - linked     : True if tgt_id is linked from src_id in the graph
    """
    if title not in title_to_id:
        return None

    node_id = title_to_id[title]
    if node_id not in id_to_idx:
        return None

    idx = id_to_idx[node_id]
    sims = sim_matrix[idx]
    top_indices = np.argsort(sims)[::-1][:top_k]

    results = []
    for i in top_indices:
        rec_id = node_ids_arr[i]
        if rec_id == node_id:
            continue
        results.append({
            "title": id_to_title[rec_id],
            "similarity": round(float(sims[i]), 6),
            "linked": is_linked(node_id, rec_id),
        })

    return results[:top_k]


# Example
test_title = list(title_to_id.keys())[0]
recs = get_recommendations_by_title(test_title)
print(f"Recommendations for '{test_title}':")
display(pd.DataFrame(recs))


## 4. Get Top 10 for X Articles (Single Table)

In [ ]:
def get_top_10_for_articles(titles):
    """
    Build a table of top-10 recommendations for each article in `titles`.
    Each row shows: Query Article, and for each Top-k: (title, similarity, linked).
    """
    rows = []
    for title in titles:
        recs = get_recommendations_by_title(title, top_k=10)
        if recs is None:
            print(f"Warning: Article '{title}' not found.")
            continue
        row = {"Query Article": title}
        for i, res in enumerate(recs):
            link_flag = " [LINKED]" if res["linked"] else ""
            row[f"Top {i+1}"] = f"{res['title']} ({res['similarity']:.4f}){link_flag}"
        rows.append(row)
    return pd.DataFrame(rows)


# Test with first 20 articles
sample_titles = list(title_to_id.keys())[:20]
df_batch = get_top_10_for_articles(sample_titles)
display(df_batch)
